# slide2pptx: two-level box detection on Colab (U-Net + pair CNN)

**Runtime → Change runtime type → T4 GPU.** No dataset needed: slides are generated on the
fly by `boxseg/synth.py`. See `docs/two_level.md` for the idea.

**Built to survive disconnects.** Training runs as a background job (`tools/colab_job.py`),
not inside the cell, so closing or reloading this page does not stop it. Checkpoints with the
optimiser state go to Drive every 250 steps, and the log is on Drive too.

* **Page lost its connection / you closed the tab:** reopen, run cell 1 and the **status** cell.
  The job is still running.
* **Runtime was reset (VM gone):** run cells 1–3 again. Cell 3 restarts the job, every stage
  continues from its last checkpoint, and finished stages are skipped.

In [ ]:
# 1. Settings
REPO_URL  = "https://github.com/C00p1r/slide2pptx.git"
BRANCH    = "claude/project-thread-jkuii6"
DRIVE_DIR = "/content/drive/MyDrive/slide2pptx"
OUT       = f"{DRIVE_DIR}/runs/boxseg"      # checkpoints, logs and reports
SIZE      = "512x288"                        # generated slide size (16:9)
SEG_STEPS, GROUP_STEPS, N_EVAL = 8000, 4000, 300
WORKERS   = 2                                # data-generating CPU processes (Colab has 2 cores)
REAL      = f"{DRIVE_DIR}/data/real/images"  # NotebookLM slides, if you have them on Drive
JOB       = f"python tools/colab_job.py --dir {OUT}"

In [ ]:
# 2. Mount Drive, get the branch, install (about 1 min)
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/slide2pptx'):
    !git clone -q -b {BRANCH} {REPO_URL} /content/slide2pptx
else:
    !git -C /content/slide2pptx fetch -q origin {BRANCH} && git -C /content/slide2pptx checkout -q {BRANCH} && git -C /content/slide2pptx pull -q
%cd /content/slide2pptx
!pip install -q opencv-python-headless
!apt-get -qq install -y fonts-noto-cjk fonts-liberation > /dev/null
!mkdir -p {OUT}
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python -m boxseg.synth --n 4 --out /content/samples.png
from IPython.display import Image as Show; display(Show('/content/samples.png', width=900))

**3. Start the whole pipeline as one background job.** Stages run in order:
Level 1 U-Net → Level 2 pair CNN → held-out-style U-Net (no hand-drawn, no sticky) →
evaluations → real-slide predictions. Running this cell again while the job runs does nothing,
so it is safe to re-run after a reconnect.

In [ ]:
T = f"python -m boxseg.train --out {OUT} --size {SIZE} --workers {WORKERS}"
E = f"python -m boxseg.evaluate --n {N_EVAL}"
steps = [
    f"{T} seg   --steps {SEG_STEPS} --batch 16",
    f"{T} group --steps {GROUP_STEPS} --batch 64",
    f"{E} --seg {OUT}/seg.pt --group {OUT}/group.pt",
    f"{T} seg   --steps {SEG_STEPS} --batch 16 --name seg_no_hand --exclude-styles hand sticky",
    f"{E} --seg {OUT}/seg_no_hand.pt --only-styles hand sticky --report {OUT}/heldout_no_hand.md",
    f"{E} --seg {OUT}/seg.pt         --only-styles hand sticky --report {OUT}/heldout_full.md",
    f"if [ -d {REAL} ]; then python -m boxseg.predict {OUT}/seg.pt {REAL} --group {OUT}/group.pt --out {OUT}/pred_real --baseline; fi",
]
pipeline = " && ".join(steps)
!{JOB} start boxseg -- "{pipeline}"

In [ ]:
# Status: is it running, and the last log lines (re-run any time)
!{JOB} status boxseg --lines 20

In [ ]:
# Results so far
from pathlib import Path
for f in ["seg_eval.md", "heldout_no_hand.md", "heldout_full.md"]:
    p = Path(OUT) / f
    if p.exists():
        print(p.read_text())

In [ ]:
# Stop the job (checkpoints stay; cell 3 continues from them)
# !{JOB} stop boxseg

Running one stage by hand, e.g. a quick test before the long job (`--resume no` starts over):

```
!python -m boxseg.train seg --steps 200 --batch 16 --out /content/test --resume no
```
From the Colab terminal the same commands work without the `!`.